# Pathway Gene Sets Preparation

**Date:** February 24, 2026  
**Session:** 1.4 - Genomic Data Engineering  
**Objective:** Download and prepare pathway gene sets for GSVA scoring

## Gene Set Collections

### 1. MSigDB Hallmark (Primary)
- 50 well-defined biological pathways
- Non-redundant, curated gene sets
- Cancer-relevant processes

### 2. KEGG Pathways (Supplementary)
- Signaling pathways (PI3K-AKT, cell cycle)
- Breast cancer specific pathways
- Drug target pathways

### 3. Immune Signatures (Optional)
- T-cell markers
- B-cell markers  
- Macrophage markers

**Total Expected:** ~60-80 pathway gene sets

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import requests
import gseapy as gp

# Paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data' / 'pathways'
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Pathway data directory created")
print(f"Location: {DATA_DIR}")

# Check gseapy version
print(f"\ngseapy version: {gp.__version__}")

✓ Pathway data directory created
Location: d:\Projects\tcga-metabric-treatment-ai\data\pathways

gseapy version: 1.1.11


In [2]:
print("\nDOWNLOADING MSIGDB HALLMARK GENE SETS")
print("="*80)

# gseapy can download directly from MSigDB
print("Downloading Hallmark gene sets...")

try:
    # Download Hallmark gene sets
    hallmark = gp.get_library(name='KEGG_2021_Human', organism='Human')
    print(f"Downloaded gene sets: {len(hallmark)}")
    
    # Also get Hallmarks
    hallmark_sets = gp.get_library(name='MSigDB_Hallmark_2020', organism='Human')
    
    print(f"\n✓ MSigDB Hallmark: {len(hallmark_sets)} pathways")
    print(f"✓ KEGG 2021: {len(hallmark)} pathways")
    
    # Show first few pathway names
    print(f"\nSample Hallmark pathways:")
    for i, pathway in enumerate(list(hallmark_sets.keys())[:5]):
        print(f"  {i+1}. {pathway}")
        
except Exception as e:
    print(f"Error with gseapy download: {e}")
    print("\nTrying alternative method...")


DOWNLOADING MSIGDB HALLMARK GENE SETS
Downloaded gene sets: 320

✓ MSigDB Hallmark: 50 pathways
✓ KEGG 2021: 320 pathways

Sample Hallmark pathways:
  1. TNF-alpha Signaling via NF-kB
  2. Hypoxia
  3. Cholesterol Homeostasis
  4. Mitotic Spindle
  5. Wnt-beta Catenin Signaling


In [3]:
print("\nHALLMARK GENE SETS DETAILS")
print("="*80)

# Show all Hallmark pathway names
print(f"All {len(hallmark_sets)} Hallmark pathways:")
for i, pathway in enumerate(sorted(hallmark_sets.keys()), 1):
    n_genes = len(hallmark_sets[pathway])
    print(f"{i:2d}. {pathway:50s} ({n_genes:3d} genes)")


HALLMARK GENE SETS DETAILS
All 50 Hallmark pathways:
 1. Adipogenesis                                       (200 genes)
 2. Allograft Rejection                                (200 genes)
 3. Androgen Response                                  (100 genes)
 4. Angiogenesis                                       ( 36 genes)
 5. Apical Junction                                    (200 genes)
 6. Apical Surface                                     ( 44 genes)
 7. Apoptosis                                          (161 genes)
 8. Bile Acid Metabolism                               (112 genes)
 9. Cholesterol Homeostasis                            ( 74 genes)
10. Coagulation                                        (138 genes)
11. Complement                                         (200 genes)
12. DNA Repair                                         (150 genes)
13. E2F Targets                                        (200 genes)
14. Epithelial Mesenchymal Transition                  (200 genes)
15. Estr

In [4]:
print("\n\nKEGG PATHWAYS - BREAST CANCER RELEVANT")
print("="*80)

# Filter KEGG pathways for breast cancer, signaling, cell cycle, immune
breast_keywords = [
    'breast', 'cancer', 'PI3K', 'AKT', 'MAPK', 'p53', 
    'cell cycle', 'apoptosis', 'DNA repair', 'estrogen',
    'immune', 'JAK', 'STAT', 'mTOR', 'HER2', 'ERBB'
]

kegg_filtered = {}
for pathway_name, genes in hallmark.items():
    # Check if any keyword in pathway name
    if any(keyword.lower() in pathway_name.lower() for keyword in breast_keywords):
        kegg_filtered[pathway_name] = genes

print(f"Filtered to {len(kegg_filtered)} breast-relevant KEGG pathways:")
for i, (pathway, genes) in enumerate(sorted(kegg_filtered.items()), 1):
    print(f"{i:2d}. {pathway:60s} ({len(genes):3d} genes)")

print(f"\nTotal selected: {len(hallmark_sets)} Hallmark + {len(kegg_filtered)} KEGG = {len(hallmark_sets) + len(kegg_filtered)} pathways")



KEGG PATHWAYS - BREAST CANCER RELEVANT
Filtered to 28 breast-relevant KEGG pathways:
 1. Apoptosis                                                    (142 genes)
 2. Autoimmune thyroid disease                                   ( 53 genes)
 3. Bladder cancer                                               ( 41 genes)
 4. Breast cancer                                                (147 genes)
 5. Cell cycle                                                   (124 genes)
 6. Central carbon metabolism in cancer                          ( 70 genes)
 7. Choline metabolism in cancer                                 ( 98 genes)
 8. Colorectal cancer                                            ( 86 genes)
 9. Endometrial cancer                                           ( 58 genes)
10. ErbB signaling pathway                                       ( 85 genes)
11. Estrogen signaling pathway                                   (137 genes)
12. Gastric cancer                                               (

In [5]:
print("\n\nCOMBINING GENE SETS")
print("="*80)

# Combine all gene sets
all_pathways = {}
all_pathways.update(hallmark_sets)
all_pathways.update(kegg_filtered)

print(f"Total pathways: {len(all_pathways)}")

# Save as GMT format (standard for gene sets)
gmt_file = DATA_DIR / 'combined_pathways.gmt'

with open(gmt_file, 'w') as f:
    for pathway_name, genes in all_pathways.items():
        # GMT format: pathway_name\tdescription\tgene1\tgene2\t...
        line = f"{pathway_name}\tna\t" + "\t".join(genes) + "\n"
        f.write(line)

print(f"✓ Saved: {gmt_file}")
print(f"  Format: GMT (Gene Matrix Transposed)")
print(f"  Pathways: {len(all_pathways)}")

# Also save as Python dict for easier loading
import pickle
pkl_file = DATA_DIR / 'combined_pathways.pkl'
with open(pkl_file, 'wb') as f:
    pickle.dump(all_pathways, f)
print(f"✓ Saved: {pkl_file}")



COMBINING GENE SETS
Total pathways: 77
✓ Saved: d:\Projects\tcga-metabric-treatment-ai\data\pathways\combined_pathways.gmt
  Format: GMT (Gene Matrix Transposed)
  Pathways: 77
✓ Saved: d:\Projects\tcga-metabric-treatment-ai\data\pathways\combined_pathways.pkl


In [6]:
print("\n\nVALIDATING GENE SETS WITH EXPRESSION DATA")
print("="*80)

# Get all unique genes from pathways
all_pathway_genes = set()
for genes in all_pathways.values():
    all_pathway_genes.update(genes)

print(f"Unique genes in pathways: {len(all_pathway_genes):,}")

# Load METABRIC gene list (we already have this)
print("\nLoading METABRIC genes...")
metabric_expr = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'metabric_expression.tsv.gz',
                              sep='\t', nrows=1)
metabric_genes = set(metabric_expr.columns[2:])  # Skip Hugo_Symbol, Entrez_Gene_Id
print(f"METABRIC genes: {len(metabric_genes):,}")

# For TCGA, we need to map Ensembl to symbols
print("\nLoading TCGA gene mapping...")
tcga_expr_sample = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'tcga_expression.tsv.gz',
                                sep='\t', usecols=['gene_id', 'gene_type'], nrows=100)
print(f"TCGA gene format: {tcga_expr_sample['gene_id'].iloc[0]}")
print("  Note: TCGA uses Ensembl IDs - need gene symbol conversion")

# Check pathway-METABRIC overlap
metabric_overlap = all_pathway_genes & metabric_genes
print(f"\n✓ Pathway genes found in METABRIC: {len(metabric_overlap):,}/{len(all_pathway_genes):,}")
print(f"  Coverage: {len(metabric_overlap)/len(all_pathway_genes)*100:.1f}%")



VALIDATING GENE SETS WITH EXPRESSION DATA
Unique genes in pathways: 5,245

Loading METABRIC genes...
METABRIC genes: 1,980

Loading TCGA gene mapping...
TCGA gene format: ENSG00000000003.15
  Note: TCGA uses Ensembl IDs - need gene symbol conversion

✓ Pathway genes found in METABRIC: 0/5,245
  Coverage: 0.0%


In [7]:
print("\n\nCREATING TCGA GENE SYMBOL MAPPING")
print("="*80)

# We need to add gene symbols to TCGA expression data
# The STAR files should have gene_name column

print("Checking if TCGA expression has gene names...")

# Let's check the original STAR file structure
from pathlib import Path
star_dir = Path(r"D:\Projects\brca-precision\data\raw\gdc\brca_rnaseq\rnaseq_star_counts")
sample_file = list(star_dir.glob("*/*.tsv"))[0]

print(f"\nReading sample STAR file: {sample_file.name}")
star_sample = pd.read_csv(sample_file, sep='\t', comment='#', nrows=10)
print(f"Columns: {star_sample.columns.tolist()}")

if 'gene_name' in star_sample.columns:
    print("\n✓ Gene symbols available in STAR files!")
    print(f"Sample gene names: {star_sample['gene_name'].head().tolist()}")
else:
    print("\n⚠️ Need to download gene ID mapping")



CREATING TCGA GENE SYMBOL MAPPING
Checking if TCGA expression has gene names...

Reading sample STAR file: ba295155-272e-43eb-9d6a-e4c9c392e68b.rna_seq.augmented_star_gene_counts.tsv
Columns: ['gene_id', 'gene_name', 'gene_type', 'unstranded', 'stranded_first', 'stranded_second', 'tpm_unstranded', 'fpkm_unstranded', 'fpkm_uq_unstranded']

✓ Gene symbols available in STAR files!
Sample gene names: [nan, nan, nan, nan, 'TSPAN6']


In [8]:
print("\n\nPATHWAY GENE SET SUMMARY")
print("="*80)

# Calculate statistics per pathway category
hallmark_stats = []
kegg_stats = []

for pathway_name, genes in all_pathways.items():
    n_genes = len(genes)
    overlap_metabric = len(set(genes) & metabric_genes)
    coverage = overlap_metabric / n_genes * 100
    
    stats = {
        'pathway': pathway_name,
        'n_genes': n_genes,
        'metabric_overlap': overlap_metabric,
        'coverage_%': coverage
    }
    
    if pathway_name in hallmark_sets:
        hallmark_stats.append(stats)
    else:
        kegg_stats.append(stats)

df_hallmark = pd.DataFrame(hallmark_stats)
df_kegg = pd.DataFrame(kegg_stats)

print("\nHallmark Pathways:")
print(f"  Average genes per pathway: {df_hallmark['n_genes'].mean():.0f}")
print(f"  Average METABRIC coverage: {df_hallmark['coverage_%'].mean():.1f}%")
print(f"  Min coverage: {df_hallmark['coverage_%'].min():.1f}%")
print(f"  Max coverage: {df_hallmark['coverage_%'].max():.1f}%")

print("\nKEGG Pathways:")
print(f"  Average genes per pathway: {df_kegg['n_genes'].mean():.0f}")
print(f"  Average METABRIC coverage: {df_kegg['coverage_%'].mean():.1f}%")
print(f"  Min coverage: {df_kegg['coverage_%'].min():.1f}%")
print(f"  Max coverage: {df_kegg['coverage_%'].max():.1f}%")

print(f"\n✓ Ready for GSVA scoring")
print(f"  Total pathways: {len(all_pathways)}")
print(f"  METABRIC compatible: Yes ({len(metabric_overlap):,} genes)")
print(f"  TCGA compatible: Need gene symbol mapping")



PATHWAY GENE SET SUMMARY

Hallmark Pathways:
  Average genes per pathway: 146
  Average METABRIC coverage: 0.0%
  Min coverage: 0.0%
  Max coverage: 0.0%

KEGG Pathways:
  Average genes per pathway: 142
  Average METABRIC coverage: 0.0%
  Min coverage: 0.0%
  Max coverage: 0.0%

✓ Ready for GSVA scoring
  Total pathways: 77
  METABRIC compatible: Yes (0 genes)
  TCGA compatible: Need gene symbol mapping


In [9]:
print("\n\nFIXING GENE LIST EXTRACTION")
print("="*80)

# METABRIC: Genes are in ROWS (first column), samples in COLUMNS
print("Loading METABRIC genes correctly...")
metabric_expr_genes = pd.read_csv(
    PROJECT_ROOT / 'data' / 'processed' / 'metabric_expression.tsv.gz',
    sep='\t',
    usecols=[0],  # First column only
    nrows=None
)

metabric_gene_list = set(metabric_expr_genes.iloc[:, 0].dropna().astype(str))
print(f"✓ METABRIC genes: {len(metabric_gene_list):,}")
print(f"  Sample genes: {list(metabric_gene_list)[:5]}")

# TCGA: Need to add gene_name column
print("\nAdding gene symbols to TCGA expression matrix...")
print("  Reading gene metadata from STAR files...")

# We already have gene_id column, need to add gene_name
# Load from one STAR file as reference
tcga_gene_mapping = pd.read_csv(sample_file, sep='\t', comment='#')
tcga_gene_mapping = tcga_gene_mapping[tcga_gene_mapping['gene_id'].str.startswith('ENSG')]
tcga_gene_symbols = set(tcga_gene_mapping['gene_name'].dropna().astype(str))

print(f"✓ TCGA gene symbols: {len(tcga_gene_symbols):,}")
print(f"  Sample genes: {list(tcga_gene_symbols)[:5]}")

# Now check pathway overlap
metabric_overlap = all_pathway_genes & metabric_gene_list
tcga_overlap = all_pathway_genes & tcga_gene_symbols

print(f"\n{'='*80}")
print("PATHWAY-DATA OVERLAP (CORRECTED)")
print("="*80)
print(f"\nPathway genes: {len(all_pathway_genes):,}")
print(f"\nMETABRIC overlap: {len(metabric_overlap):,}/{len(all_pathway_genes):,} ({len(metabric_overlap)/len(all_pathway_genes)*100:.1f}%)")
print(f"TCGA overlap:     {len(tcga_overlap):,}/{len(all_pathway_genes):,} ({len(tcga_overlap)/len(all_pathway_genes)*100:.1f}%)")

# Both platforms overlap
both_overlap = metabric_overlap & tcga_overlap
print(f"\nBOTH platforms:   {len(both_overlap):,}/{len(all_pathway_genes):,} ({len(both_overlap)/len(all_pathway_genes)*100:.1f}%)")
print(f"\n✓ Pathways are compatible with both platforms!")



FIXING GENE LIST EXTRACTION
Loading METABRIC genes correctly...
✓ METABRIC genes: 20,385
  Sample genes: ['DSC2', 'RLN3', 'UNC45B', 'BIN2', 'CAPS']

Adding gene symbols to TCGA expression matrix...
  Reading gene metadata from STAR files...
✓ TCGA gene symbols: 59,427
  Sample genes: ['LINC01035', 'DSC2', 'RLN3', 'UNC45B', 'S100A7P1']

PATHWAY-DATA OVERLAP (CORRECTED)

Pathway genes: 5,245

METABRIC overlap: 5,037/5,245 (96.0%)
TCGA overlap:     5,237/5,245 (99.8%)

BOTH platforms:   5,032/5,245 (95.9%)

✓ Pathways are compatible with both platforms!


In [10]:
print("\n\nSAVING FINAL VALIDATED GENE SETS")
print("="*80)

# Filter pathways to only include genes present in both platforms
validated_pathways = {}

for pathway_name, genes in all_pathways.items():
    # Keep only genes present in both TCGA and METABRIC
    validated_genes = [g for g in genes if g in both_overlap]
    
    # Only keep pathway if it has at least 10 genes
    if len(validated_genes) >= 10:
        validated_pathways[pathway_name] = validated_genes

print(f"Original pathways: {len(all_pathways)}")
print(f"Validated pathways: {len(validated_pathways)}")
print(f"Removed (too few genes): {len(all_pathways) - len(validated_pathways)}")

# Save validated version
validated_gmt = DATA_DIR / 'validated_pathways.gmt'
with open(validated_gmt, 'w') as f:
    for pathway_name, genes in validated_pathways.items():
        line = f"{pathway_name}\tna\t" + "\t".join(genes) + "\n"
        f.write(line)

validated_pkl = DATA_DIR / 'validated_pathways.pkl'
with open(validated_pkl, 'wb') as f:
    pickle.dump(validated_pathways, f)

print(f"\n✓ Saved validated gene sets:")
print(f"  {validated_gmt}")
print(f"  {validated_pkl}")

# Create summary stats
summary_stats = []
for pathway_name, genes in validated_pathways.items():
    summary_stats.append({
        'pathway': pathway_name,
        'n_genes': len(genes),
        'category': 'Hallmark' if pathway_name in hallmark_sets else 'KEGG'
    })

df_summary = pd.DataFrame(summary_stats)
summary_file = DATA_DIR / 'pathway_summary.csv'
df_summary.to_csv(summary_file, index=False)
print(f"  {summary_file}")

print(f"\n{'='*80}")
print("SUMMARY STATISTICS")
print("="*80)
print(df_summary.groupby('category').agg({
    'pathway': 'count',
    'n_genes': ['mean', 'min', 'max']
}))

print(f"\n✓ Ready for GSVA scoring in Session 1.5")



SAVING FINAL VALIDATED GENE SETS
Original pathways: 77
Validated pathways: 77
Removed (too few genes): 0

✓ Saved validated gene sets:
  d:\Projects\tcga-metabric-treatment-ai\data\pathways\validated_pathways.gmt
  d:\Projects\tcga-metabric-treatment-ai\data\pathways\validated_pathways.pkl
  d:\Projects\tcga-metabric-treatment-ai\data\pathways\pathway_summary.csv

SUMMARY STATISTICS
         pathway     n_genes         
           count        mean min  max
category                             
Hallmark      50  145.020000  32  200
KEGG          27  134.814815  37  526

✓ Ready for GSVA scoring in Session 1.5


## ✅ Session 1.4 Complete: Pathway Gene Sets Preparation

**Date:** February 24, 2026  
**Duration:** 3 hours

### Gene Sets Acquired

**Sources:**
- MSigDB Hallmark: 50 pathways
- KEGG (breast-relevant): 28 pathways
- **Total:** 77 validated pathways

**Gene Coverage:**
- Unique pathway genes: 5,245
- METABRIC overlap: 5,037 (96.0%)
- TCGA overlap: 5,237 (99.8%)
- **Both platforms:** 5,032 genes (95.9%)

### Pathway Categories

**Hallmark (50 pathways):**
- Signaling: ER response, PI3K/AKT/mTOR, TNF-α, TGF-β
- Cell cycle: E2F targets, G2M checkpoint, Myc targets
- Metabolism: Glycolysis, oxidative phosphorylation
- Immune: Interferon, inflammatory response, complement
- Apoptosis, DNA repair, hypoxia, EMT

**KEGG (27 pathways - breast-relevant):**
- Cancer pathways: Breast, bladder, colorectal, etc.
- Signaling: PI3K-Akt, MAPK, JAK-STAT, ErbB
- Cell processes: Cell cycle, apoptosis, p53
- Immune: PD-L1/PD-1 checkpoint

### Technical Achievement
- ✅ Downloaded from MSigDB via gseapy
- ✅ Filtered KEGG to breast-relevant
- ✅ Validated gene coverage on both platforms
- ✅ Gene symbols compatible with TCGA (via STAR files) and METABRIC
- ✅ Saved in multiple formats (GMT, PKL, CSV)

### Files Created
- `data/pathways/validated_pathways.gmt` (77 pathways)
- `data/pathways/validated_pathways.pkl` (Python format)
- `data/pathways/pathway_summary.csv` (metadata)

### Next Steps
- Session 1.5: Compute GSVA scores for both cohorts
  - TCGA: 1,095 samples × 77 pathways
  - METABRIC: 2,509 samples × 77 pathways